# NB-03 | Feature Engineering

**Project:** GA4 Google Merchandise Store — Conversion Propensity Scoring  
**Methodology:** CRISP-DM — Data Preparation phase  
**Input:** `data/raw/sessions.csv` (360,129 rows × 13 cols)  
**Output:** `data/processed/` — Parquet splits ready for NB-04 modelling

---

## Objectives

Transform the raw session-level CSV into a clean, model-ready feature matrix by:

1. Dropping identifier columns (`user_pseudo_id`, `session_id`)
2. Capping outliers in `session_duration_sec` at the confirmed p99 = 2,456 sec
3. Engineering two funnel-efficiency ratio features
4. Encoding categorical variables (`device`, `traffic_medium`, `traffic_source`, `country`)
5. Splitting into train / test sets (80/20, stratified)
6. Saving all artefacts as Parquet

All decisions are grounded in EDA findings from NB-02.

## 0 — Imports & configuration

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# Reproducibility
RANDOM_STATE = 42

# Paths
RAW_DIR       = Path('data/raw')
PROCESSED_DIR = Path('data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Confirmed from NB-02 EDA
P99_DURATION   = 2456          # session_duration_sec p99
TOP_N_COUNTRIES = 10           # keep top-10 countries, rest → 'other'
SCALE_POS_WEIGHT = 73.3        # for XGBoost reference (documented here)

print('Libraries loaded.')
print(f'Output directory: {PROCESSED_DIR.resolve()}')

Libraries loaded.
Output directory: /Users/seihavat/Documents/Personal Project/GA4 Machandise Store Project/data/processed


## 1 — Load raw data

In [2]:
sessions = pd.read_csv(RAW_DIR / 'sessions.csv')

print(f'Shape: {sessions.shape}')
print(f'\nColumns: {sessions.columns.tolist()}')
sessions.head(3)

Shape: (360129, 13)

Columns: ['user_pseudo_id', 'session_id', 'converted', 'total_events', 'page_views', 'items_viewed', 'add_to_cart', 'checkout_starts', 'session_duration_sec', 'device', 'country', 'traffic_medium', 'traffic_source']


,user_pseudo_id,session_id,converted,total_events,page_views,items_viewed,add_to_cart,checkout_starts,session_duration_sec,device,country,traffic_medium,traffic_source
0,1.000534e+07,1498493162,0,4,1,0,0,0,1147.73,desktop,China,(data deleted),(data deleted)
1,1.000534e+07,3552270955,0,2,0,0,0,0,0.00,desktop,China,(data deleted),(data deleted)
2,1.000706e+07,4483881219,0,9,3,0,0,0,80.93,mobile,Japan,(data deleted),(data deleted)


In [3]:
# Sanity check against NB-02 confirmed figures
assert sessions.shape[0] == 360_129, f'Expected 360,129 rows — got {sessions.shape[0]}'
assert sessions['converted'].sum() == 4_848, f'Expected 4,848 conversions — got {sessions["converted"].sum()}'
assert sessions.isnull().sum().sum() == 0, 'Unexpected nulls in sessions.csv'

print('✓ Row count matches NB-02 (360,129)')
print(f'✓ Conversions: {sessions["converted"].sum():,}  ({sessions["converted"].mean()*100:.2f}%)')
print('✓ No null values')

✓ Row count matches NB-02 (360,129)
✓ Conversions: 4,848  (1.35%)
✓ No null values


## 2 — Drop identifier columns

`user_pseudo_id` and `session_id` are surrogate keys — they carry no predictive signal and must be removed before feature engineering.

In [4]:
ID_COLS = ['user_pseudo_id', 'session_id']
sessions = sessions.drop(columns=ID_COLS)

print(f'Shape after dropping ID columns: {sessions.shape}')
print(f'Remaining columns: {sessions.columns.tolist()}')

Shape after dropping ID columns: (360129, 11)
Remaining columns: ['converted', 'total_events', 'page_views', 'items_viewed', 'add_to_cart', 'checkout_starts', 'session_duration_sec', 'device', 'country', 'traffic_medium', 'traffic_source']


## 3 — Cap outliers in `session_duration_sec`

**Rationale (from NB-02):** The maximum value is 402,477 sec (≈ 112 hours) — clearly bot or stuck-tab traffic. The p99 is 2,456 sec (≈ 41 min), which represents the plausible upper bound for a genuine shopping session. Capping at p99 preserves the distribution for the vast majority of users while neutralising extreme outliers that would distort tree splits.

Count-based features (`total_events`, `page_views`, etc.) are **not** capped — tree-based models handle these naturally, and the extreme values are far less extreme in magnitude.

In [5]:
# Confirm p99 matches NB-02 before applying cap
actual_p99 = sessions['session_duration_sec'].quantile(0.99)
print(f'Computed p99: {actual_p99:,.0f} sec  |  Expected: {P99_DURATION} sec')

sessions['session_duration_sec'] = sessions['session_duration_sec'].clip(upper=P99_DURATION)

print(f'\nAfter cap:')
print(f'  Max : {sessions["session_duration_sec"].max():,.0f} sec')
print(f'  Mean: {sessions["session_duration_sec"].mean():,.1f} sec')
print(f'  Std : {sessions["session_duration_sec"].std():,.1f} sec')

Computed p99: 2,456 sec  |  Expected: 2456 sec

After cap:
  Max : 2,456 sec
  Mean: 188.7 sec
  Std : 436.1 sec


## 4 — Derived ratio features

Two funnel-efficiency ratios that capture *quality* of engagement, not just volume:

| Feature | Formula | Interpretation |
|---|---|---|
| `cart_to_view_ratio` | `add_to_cart / items_viewed` | Of items viewed, what fraction was added to cart? |
| `checkout_to_cart_ratio` | `checkout_starts / add_to_cart` | Of cart additions, what fraction proceeded to checkout? |

**Zero-denominator handling:** Replace 0 denominators with `NaN` before dividing, then `fillna(0)`. A session with zero `items_viewed` has a *meaningfully zero* cart-to-view ratio — it is not missing data.

**EDA motivation:** NB-02 showed that 80.3% of sessions that viewed a product did not add to cart. These ratios should be among the strongest predictors of conversion.

In [9]:
sessions['cart_to_view_ratio'] = (
    sessions['add_to_cart'] / sessions['items_viewed'].replace(0, np.nan)
).fillna(0)

sessions['checkout_to_cart_ratio'] = (
    sessions['checkout_starts'] / sessions['add_to_cart'].replace(0, np.nan)
).fillna(0)

# Validation
assert sessions['cart_to_view_ratio'].isnull().sum() == 0
assert sessions['checkout_to_cart_ratio'].isnull().sum() == 0

print('cart_to_view_ratio')
print(sessions['cart_to_view_ratio'].describe().round(4))
print()
print('checkout_to_cart_ratio')
print(sessions['checkout_to_cart_ratio'].describe().round(4))

cart_to_view_ratio
count    360129.0000
mean          0.0275
std           0.1645
min           0.0000
25%           0.0000
50%           0.0000
75%           0.0000
max          10.0000
Name: cart_to_view_ratio, dtype: float64

checkout_to_cart_ratio
count    360129.0000
mean          0.0298
std           0.3602
min           0.0000
25%           0.0000
50%           0.0000
75%           0.0000
max          42.0000
Name: checkout_to_cart_ratio, dtype: float64


In [10]:
# Sense check: sessions with checkout_starts should have higher ratios than those without
mask_checkout = sessions['checkout_starts'] > 0
print(f'Sessions with checkout_starts > 0: {mask_checkout.sum():,}')
print(f'  Mean checkout_to_cart_ratio (checkout sessions): {sessions.loc[mask_checkout, "checkout_to_cart_ratio"].mean():.3f}')
print(f'  Mean checkout_to_cart_ratio (no checkout):       {sessions.loc[~mask_checkout, "checkout_to_cart_ratio"].mean():.3f}')

Sessions with checkout_starts > 0: 11,106
  Mean checkout_to_cart_ratio (checkout sessions): 0.966
  Mean checkout_to_cart_ratio (no checkout):       0.000


## 5 — Encode `device`

Three levels: `desktop`, `mobile`, `tablet`. Label-encoded as 0 / 1 / 2.  
Ordinal encoding is acceptable here — tree models don't assume ordinality, and it avoids adding 2 extra columns for 3 levels.

In [12]:
print('device value counts (pre-encoding):')
print(sessions['device'].value_counts())

device_map = {'desktop': 0, 'mobile': 1, 'tablet': 2}
sessions['device'] = sessions['device'].map(device_map)

assert sessions['device'].isnull().sum() == 0, 'Unmapped device values produced NaN'
print('\ndevice value counts (post-encoding):')
print(sessions['device'].value_counts().sort_index())

device value counts (pre-encoding):
device
desktop    208942
mobile     143185
tablet       8002
Name: count, dtype: int64

device value counts (post-encoding):
device
0    208942
1    143185
2      8002
Name: count, dtype: int64


## 6 — One-hot encode `traffic_medium`

Six unique values confirmed in NB-02: `organic`, `(none)`, `referral`, `<Other>`, `(data deleted)`, `cpc`.

`drop_first=False` — retain all 6 levels. The missing-value-like strings (`<Other>`, `(data deleted)`) are legitimate categories in the obfuscated dataset. Keeping all levels maximises SHAP interpretability in NB-06.

In [13]:
print('traffic_medium value counts:')
print(sessions['traffic_medium'].value_counts())

medium_dummies = pd.get_dummies(sessions['traffic_medium'], prefix='medium', drop_first=False)
print(f'\nNew columns ({len(medium_dummies.columns)}): {medium_dummies.columns.tolist()}')

sessions = pd.concat([sessions.drop(columns=['traffic_medium']), medium_dummies], axis=1)

traffic_medium value counts:
traffic_medium
organic           122841
(none)             83459
referral           63524
<Other>            52058
(data deleted)     22629
cpc                15618
Name: count, dtype: int64

New columns (6): ['medium_(data deleted)', 'medium_(none)', 'medium_<Other>', 'medium_cpc', 'medium_organic', 'medium_referral']


## 7 — One-hot encode `traffic_source`

Five unique values confirmed in NB-02 — low enough cardinality to encode all levels directly.

In [14]:
print('traffic_source value counts:')
print(sessions['traffic_source'].value_counts())

source_dummies = pd.get_dummies(sessions['traffic_source'], prefix='source', drop_first=False)
print(f'\nNew columns ({len(source_dummies.columns)}): {source_dummies.columns.tolist()}')

sessions = pd.concat([sessions.drop(columns=['traffic_source']), source_dummies], axis=1)

traffic_source value counts:
traffic_source
google                             128296
<Other>                             97289
(direct)                            83459
shop.googlemerchandisestore.com     28849
(data deleted)                      22236
Name: count, dtype: int64

New columns (5): ['source_(data deleted)', 'source_(direct)', 'source_<Other>', 'source_google', 'source_shop.googlemerchandisestore.com']


## 8 — Top-10 country encoding

**Problem:** 109 unique countries — one-hot encoding all would add 109 sparse columns, most near-zero.  
**Solution:** Keep the top-10 by session volume (confirmed in NB-02); bucket everything else as `other`.

Top-10 confirmed: United States, India, Canada, United Kingdom, France, Spain, Germany, China, Taiwan, Italy.

In [15]:
# Derive top-10 from data (do not hard-code the names — let the data confirm them)
top_countries = sessions['country'].value_counts().head(TOP_N_COUNTRIES).index.tolist()
print(f'Top {TOP_N_COUNTRIES} countries:')
for i, c in enumerate(top_countries, 1):
    n = sessions['country'].value_counts()[c]
    pct = n / len(sessions) * 100
    print(f'  {i:2d}. {c:<20s}  {n:>7,}  ({pct:.1f}%)')

# Bucket all others
sessions['country_grouped'] = sessions['country'].where(
    sessions['country'].isin(top_countries), other='other'
)

print(f'\nCountries mapped to "other": {sessions["country"].nunique() - TOP_N_COUNTRIES}')
print(f'Sessions in "other" bucket: {(sessions["country_grouped"] == "other").sum():,}')

Top 10 countries:
   1. United States         158,155  (43.9%)
   2. India                  33,769  (9.4%)
   3. Canada                 26,824  (7.4%)
   4. United Kingdom         11,327  (3.1%)
   5. France                  7,162  (2.0%)
   6. Spain                   6,667  (1.9%)
   7. Germany                 6,393  (1.8%)
   8. China                   6,258  (1.7%)
   9. Taiwan                  6,057  (1.7%)
  10. Italy                   4,998  (1.4%)

Countries mapped to "other": 99
Sessions in "other" bucket: 92,519


In [16]:
country_dummies = pd.get_dummies(sessions['country_grouped'], 
                                 prefix='country', drop_first=False)
print(f'Country dummy columns ({len(country_dummies.columns)}): {country_dummies.columns.tolist()}')

sessions = pd.concat(
    [sessions.drop(columns=['country', 'country_grouped']), country_dummies],
    axis=1
)

Country dummy columns (11): ['country_Canada', 'country_China', 'country_France', 'country_Germany', 'country_India', 'country_Italy', 'country_Spain', 'country_Taiwan', 'country_United Kingdom', 'country_United States', 'country_other']


## 9 — Final feature matrix inspection

In [17]:
print(f'Final shape: {sessions.shape}')
print(f'\nAll columns ({len(sessions.columns)}):')
for col in sessions.columns:
    print(f'  {col:<45s}  dtype={sessions[col].dtype}')

Final shape: (360129, 32)

All columns (32):
  converted                                      dtype=int64
  total_events                                   dtype=int64
  page_views                                     dtype=int64
  items_viewed                                   dtype=int64
  add_to_cart                                    dtype=int64
  checkout_starts                                dtype=int64
  session_duration_sec                           dtype=float64
  device                                         dtype=int64
  cart_to_view_ratio                             dtype=float64
  checkout_to_cart_ratio                         dtype=float64
  medium_(data deleted)                          dtype=bool
  medium_(none)                                  dtype=bool
  medium_<Other>                                 dtype=bool
  medium_cpc                                     dtype=bool
  medium_organic                                 dtype=bool
  medium_referral                      

In [18]:
# Global null check
null_counts = sessions.isnull().sum()
if null_counts.sum() == 0:
    print('✓ No null values in feature matrix')
else:
    print('⚠ Columns with nulls:')
    print(null_counts[null_counts > 0])

✓ No null values in feature matrix


## 10 — Separate target and save full feature matrix

In [19]:
y = sessions['converted']
X = sessions.drop(columns=['converted'])

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'y value counts:\n{y.value_counts()}')
print(f'\nConversion rate: {y.mean()*100:.2f}%')

# Save full feature matrix
X.to_parquet(PROCESSED_DIR / 'features.parquet', index=False)
print(f'\n✓ Saved features.parquet  ({X.shape[0]:,} rows × {X.shape[1]} cols)')

X shape: (360129, 31)
y shape: (360129,)
y value counts:
converted
0    355281
1      4848
Name: count, dtype: int64

Conversion rate: 1.35%

✓ Saved features.parquet  (360,129 rows × 31 cols)


## 11 — Train / test split

- **80 / 20** split — standard for this dataset size; leaves ~72,000 rows for test evaluation
- **`stratify=y`** — preserves the 1.35% positive rate in both splits
- **`random_state=42`** — reproducibility

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print('Split sizes:')
print(f'  X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'  X_test:  {X_test.shape}   |  y_test:  {y_test.shape}')

Split sizes:
  X_train: (288103, 31)  |  y_train: (288103,)
  X_test:  (72026, 31)   |  y_test:  (72026,)


## 12 — Verify stratification

In [21]:
train_rate = y_train.mean() * 100
test_rate  = y_test.mean()  * 100
full_rate  = y.mean()       * 100

print(f'Conversion rate — full dataset : {full_rate:.3f}%')
print(f'Conversion rate — train split  : {train_rate:.3f}%')
print(f'Conversion rate — test split   : {test_rate:.3f}%')

# Assert stratification held (within 0.1 percentage points)
assert abs(train_rate - full_rate) < 0.1, 'Train conversion rate drifted from full dataset'
assert abs(test_rate  - full_rate) < 0.1, 'Test conversion rate drifted from full dataset'
print('\n✓ Stratification confirmed — conversion rate consistent across splits')

print(f'\nClass counts:')
print(f'  Train — class 0: {(y_train==0).sum():,}  |  class 1: {(y_train==1).sum():,}')
print(f'  Test  — class 0: {(y_test==0).sum():,}   |  class 1: {(y_test==1).sum():,}')
print(f'\n  XGBoost scale_pos_weight (reference): {SCALE_POS_WEIGHT}')

Conversion rate — full dataset : 1.346%
Conversion rate — train split  : 1.346%
Conversion rate — test split   : 1.347%

✓ Stratification confirmed — conversion rate consistent across splits

Class counts:
  Train — class 0: 284,225  |  class 1: 3,878
  Test  — class 0: 71,056   |  class 1: 970

  XGBoost scale_pos_weight (reference): 73.3


## 13 — Save all splits as Parquet

In [22]:
X_train.to_parquet(PROCESSED_DIR / 'X_train.parquet', index=False)
X_test.to_parquet( PROCESSED_DIR / 'X_test.parquet',  index=False)
y_train.to_frame().to_parquet(PROCESSED_DIR / 'y_train.parquet', index=False)
y_test.to_frame().to_parquet( PROCESSED_DIR / 'y_test.parquet',  index=False)

print('Parquet files saved:')
for f in sorted(PROCESSED_DIR.glob('*.parquet')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<30s}  {size_kb:>8.1f} KB')

Parquet files saved:
  X_test.parquet                     625.3 KB
  X_train.parquet                   2251.9 KB
  features.parquet                  2456.5 KB
  y_test.parquet                       3.5 KB
  y_train.parquet                      9.8 KB


## 14 — Reload verification

Confirm all Parquet files round-trip correctly.

In [23]:
X_train_check = pd.read_parquet(PROCESSED_DIR / 'X_train.parquet')
X_test_check  = pd.read_parquet(PROCESSED_DIR / 'X_test.parquet')
y_train_check = pd.read_parquet(PROCESSED_DIR / 'y_train.parquet')
y_test_check  = pd.read_parquet(PROCESSED_DIR / 'y_test.parquet')

assert X_train_check.shape == X_train.shape
assert X_test_check.shape  == X_test.shape
assert len(y_train_check)  == len(y_train)
assert len(y_test_check)   == len(y_test)
assert X_train_check.isnull().sum().sum() == 0
assert X_test_check.isnull().sum().sum()  == 0

print('✓ X_train reloaded correctly:', X_train_check.shape)
print('✓ X_test  reloaded correctly:', X_test_check.shape)
print('✓ y_train reloaded correctly:', y_train_check.shape)
print('✓ y_test  reloaded correctly:', y_test_check.shape)
print('✓ No nulls in reloaded feature matrices')

✓ X_train reloaded correctly: (288103, 31)
✓ X_test  reloaded correctly: (72026, 31)
✓ y_train reloaded correctly: (288103, 1)
✓ y_test  reloaded correctly: (72026, 1)
✓ No nulls in reloaded feature matrices


## 15 — Feature engineering summary

Final reference table for NB-04.

In [24]:
feature_summary = pd.DataFrame({
    'feature': X_train.columns,
    'dtype':   X_train.dtypes.values,
    'mean':    X_train.mean().round(4).values,
    'std':     X_train.std().round(4).values,
    'min':     X_train.min().values,
    'max':     X_train.max().values,
})
print(feature_summary.to_string(index=False))

                               feature   dtype     mean      std   min     max
                          total_events   int64  11.9406  26.5451     1    1007
                            page_views   int64   3.7546   9.0545     0     566
                          items_viewed   int64   1.0736   4.7705     0     319
                           add_to_cart   int64   0.1640   1.3415     0      74
                       checkout_starts   int64   0.1084   0.8592     0      54
                  session_duration_sec float64 188.8378 435.8119   0.0  2456.0
                                device   int64   0.4420   0.5399     0       2
                    cart_to_view_ratio float64   0.0276   0.1648   0.0     9.0
                checkout_to_cart_ratio float64   0.0302   0.3660   0.0    42.0
                 medium_(data deleted)    bool   0.0626   0.2422 False    True
                         medium_(none)    bool   0.2316   0.4218 False    True
                        medium_<Other>    bool   0.1

In [25]:
print('=' * 60)
print('NB-03 COMPLETE — Feature Engineering Summary')
print('=' * 60)
print(f'  Input rows            : 360,129')
print(f'  Final feature count   : {X.shape[1]}')
print(f'  Train rows            : {X_train.shape[0]:,}')
print(f'  Test rows             : {X_test.shape[0]:,}')
print(f'  Train conversion rate : {y_train.mean()*100:.3f}%')
print(f'  Test  conversion rate : {y_test.mean()*100:.3f}%')
print(f'  Null values in X      : 0')
print(f'  Output directory      : data/processed/')
print()
print('Files ready for NB-04:')
print('  data/processed/X_train.parquet')
print('  data/processed/X_test.parquet')
print('  data/processed/y_train.parquet')
print('  data/processed/y_test.parquet')
print('  data/processed/features.parquet')

NB-03 COMPLETE — Feature Engineering Summary
  Input rows            : 360,129
  Final feature count   : 31
  Train rows            : 288,103
  Test rows             : 72,026
  Train conversion rate : 1.346%
  Test  conversion rate : 1.347%
  Null values in X      : 0
  Output directory      : data/processed/

Files ready for NB-04:
  data/processed/X_train.parquet
  data/processed/X_test.parquet
  data/processed/y_train.parquet
  data/processed/y_test.parquet
  data/processed/features.parquet


---

## Engineering decisions — rationale record

| Decision | Choice | Rationale |
|---|---|---|
| Outlier capping | `session_duration_sec` capped at p99 (2,456 sec) | Max of 402,477 sec is bot traffic; count features left uncapped (tree models handle them) |
| Ratio features | `cart_to_view_ratio`, `checkout_to_cart_ratio` | Capture funnel *efficiency*, not just volume; EDA showed 80.3% view-to-no-cart drop-off |
| Device encoding | Label encode (0/1/2) | 3 levels; trees don't assume ordinality; avoids 2 extra columns |
| `traffic_medium` | One-hot, all 6 levels, `drop_first=False` | Low cardinality; `<Other>` and `(data deleted)` are legitimate obfuscation categories |
| `traffic_source` | One-hot, all 5 levels, `drop_first=False` | Low cardinality; same reasoning |
| Country encoding | Top-10 + `other` bucket, then one-hot | 109 countries → 11 columns; balances signal vs sparsity |
| `drop_first=False` | Retained across all one-hot steps | Preserves all levels for SHAP interpretability in NB-06 |
| Train/test split | 80/20, stratified, `random_state=42` | Standard split; stratification preserves 1.35% positive rate |
| Artifact format | Parquet | Efficient columnar storage; preserves dtypes; standard for pandas/sklearn pipelines |

**Next notebook:** NB-04 — Model Comparison (Logistic Regression vs Random Forest vs XGBoost)